# Inspect GT data

In [ ]:
import rasterio

GT_TIFF = "lab_data/GBDA24_ex2_ref_data.tif"

with rasterio.open(GT_TIFF) as src:
    p = src.profile
    t = p.get('transform')
    
    print("Raster Information:")
    print(f"  Driver: {p.get('driver')}")
    print(f"  Shape:  {p.get('width')}w x {p.get('height')}h")
    print(f"  CRS:    {p.get('crs')}")
    
    # Beautified Transform
    # Affine elements: a, b, c, d, e, f (c and f are offsets)
    print("  Transform (Affine Matrix):")
    # Using .4g for general/scientific and .2f for standard to maintain alignment
    def fmt(val): 
        return f"{val:>12.4e}" if abs(val) < 1e-4 and val != 0 else f"{val:>12.2f}"

    print(f"    | {fmt(t.a)} {fmt(t.b)} | {fmt(t.c)} | (x_res, x_rot, x_offset)")
    print(f"    | {fmt(t.d)} {fmt(t.e)} | {fmt(t.f)} | (y_rot, y_res, y_offset)")
    print(f"    | {fmt(0.0)} {fmt(0.0)} | {fmt(1.0)} |")

    print("\nBand Information:")
    for i in range(1, src.count + 1):
        # Using masked=True is faster if you just want stats without NoData interference
        band_data = src.read(i, masked=True)
        print(f"  Band {i}:")
        print(f"    Type:    {band_data.dtype}")
        print(f"    NoData:  {src.nodatavals[i-1]}")
        print(f"    Min/Max: {band_data.min():.2f} / {band_data.max():.2f}")
        print(f"    Mean:    {band_data.mean():.2f}\n")

# Download S2 L1C data

In [ ]:
import json
import boto3
from botocore.config import Config
import os
import pystac_client
import rasterio

# --- A) Get GT raster bounds ---
# Assuming GT_TIFF is defined elsewhere in your code
with rasterio.open(GT_TIFF) as src:
    crs = src.crs
    bounds = src.bounds
    if crs.to_string() != "EPSG:4326":
        from rasterio.warp import transform_bounds

        bounds_wgs84 = transform_bounds(crs, "EPSG:4326", *bounds)
    else:
        bounds_wgs84 = bounds

# --- B) STAC Search ---
stac = pystac_client.Client.open("https://stac.dataspace.copernicus.eu/v1")
search = stac.search(
    collections=["sentinel-2-l1c"],
    bbox=list(bounds_wgs84),
    datetime="2021-05-01/2021-09-30",
    query={"eo:cloud_cover": {"lt": 5}},
    max_items=10,
)
items = list(search.items())

# --- C) Selection Logic (Least Cloud Cover) ---
tile_best = {}
for item in items:
    tile = item.properties.get("grid:code")
    cc = item.properties.get("eo:cloud_cover", 100)
    if tile and (tile not in tile_best or cc < tile_best[tile][0]):
        tile_best[tile] = (cc, item)
selected = {tile: item for tile, (cc, item) in tile_best.items()}

# --- D) Download via Requests and Convert to GeoTIFF ---


# --- 1. Load S3 Credentials ---
# Get your credentials at: https://eodata-s3keysmanager.dataspace.copernicus.eu/panel/s3-credentials
with open("lab_data/.s3.secret", "r") as f:
    creds = json.load(f)  # {"access_key": "ACCESS_KEY", "secret_key": "SECRET_KEY"}

# --- 2. Initialize S3 Client ---
# CDSE requires the endpoint_url and usually a specific path style
s3_client = boto3.client(
    "s3",
    aws_access_key_id=creds["access_key"],
    aws_secret_access_key=creds["secret_key"],
    endpoint_url="https://eodata.dataspace.copernicus.eu",
    region_name="eu-central-1",
    config=Config(s3={"addressing_style": "path"}),  # Crucial for CDSE
)

# --- 3. Download Loop ---
band_keys = {f"B{str(i).zfill(2)}" for i in range(1, 13)} | {"B8A"}

for tile, item in selected.items():
    print(f"\nProcessing Tile: {tile}")

    download_dir = f"lab_data/S2/{tile}"
    os.makedirs(download_dir, exist_ok=True)

    for asset_key, asset in item.assets.items():
        if asset_key not in band_keys:
            continue

        url = asset.href  # Expected: s3://eodata/Sentinel-2/...
        if not url.startswith("s3://"):
            continue

        # Parse bucket and key from s3:// URL
        path_parts = url.replace("s3://", "").split("/")
        bucket = path_parts[0]
        key = "/".join(path_parts[1:])

        ext = os.path.splitext(url)[-1]
        out_path = os.path.join(download_dir, f"{tile}-{asset_key}{ext}")

        print(f"  S3 GET: {asset_key} -> {os.path.basename(out_path)}")

        try:
            # Direct download to file
            s3_client.download_file(bucket, key, out_path)

        except Exception as e:
            print(f"  Error downloading {asset_key}: {e}")

print("\nRaw S3 downloads complete.")

# Inspect a tile's images

In [ ]:
import os
from pathlib import Path
import rasterio


def parse_jp2_metadata(folder_path):
    folder = Path(folder_path)
    jp2_files = sorted(list(folder.glob("*.jp2")))

    if not jp2_files:
        print(f"No .jp2 files found in {folder_path}")
        return

    # Table Header
    header = (
        f"{'File Name':<35} | {'Shape (H, W)':<15} | {'Bands':<6} | {'GSD (m)':<15}"
    )
    print("\n" + header)
    print("-" * len(header))

    for tif in jp2_files:
        try:
            with rasterio.open(tif) as src:
                # Shape and Bands
                shape = f"{src.height}, {src.width}"
                bands = src.count

                # GSD (Ground Sampling Distance)
                # S2 bands vary (10m, 20m, 60m)
                # res[0] is pixel width, res[1] is pixel height
                res_x, res_y = src.res
                gsd = f"{abs(res_x):.1f} x {abs(res_y):.1f}"

                # Elegant Print
                print(f"{tif.name:<35} | {shape:<15} | {bands:<6} | {gsd:<15}")

        except Exception as e:
            print(f"{tif.name:<35} | Error: {e}")


# Run it on your S2 data directory
parse_jp2_metadata("lab_data/S2/MGRS-34SEJ")

# Stack images to datacube for each tile!

In [ ]:
import rasterio
from rasterio.enums import Resampling
from pathlib import Path


def create_sentinel2_datacube(tile_dir, output_dir):
    tile_path = Path(tile_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Get tile name from directory or file prefix (e.g., MGRS-34SEJ)
    tif_files = sorted(
        list(tile_path.glob("*.jp2"))
    )  # or .tif depending on your download
    if not tif_files:
        print("No files found to stack.")
        return

    tile_name = tif_files[0].name.split("-B")[0]
    out_file = output_path / f"{tile_name}_datacube.tif"

    # 1. Identify a 10m band to use as the Master Grid (Reference)
    # B02 is always 10m and a safe bet.
    master_band_path = next((f for f in tif_files if "B02" in f.name), tif_files[0])

    with rasterio.open(master_band_path) as ref:
        master_profile = ref.profile.copy()
        master_width = ref.width
        master_height = ref.height
        master_transform = ref.transform

    # Define band order for the cube (Standard S2 order)
    bands_to_stack = [
        "B01",
        "B02",
        "B03",
        "B04",
        "B05",
        "B06",
        "B07",
        "B08",
        "B8A",
        "B09",
        "B10",
        "B11",
        "B12",
    ]

    # Filter only available files
    available_files = {f.name.split("-")[-1].split(".")[0]: f for f in tif_files}
    final_band_list = [b for b in bands_to_stack if b in available_files]

    # Update profile for multi-band output
    master_profile.update(
        {
            "driver": "GTiff",
            "count": len(final_band_list),
            "width": master_width,
            "height": master_height,
            "transform": master_transform,
            "compress": "lzw",
            "tiled": True,
            "blockxsize": 256,
            "blockysize": 256,
        }
    )

    print(f"Creating datacube: {out_file.name}")
    print(f"Target Shape: ({master_height}, {master_width}) at 10m GSD")

    with rasterio.open(out_file, "w", **master_profile) as dst:
        for i, band_key in enumerate(final_band_list, start=1):
            file_path = available_files[band_key]

            with rasterio.open(file_path) as src:
                # If resolution matches, read directly; otherwise, resample bilinear
                if src.width == master_width and src.height == master_height:
                    data = src.read(1)
                else:
                    print(f"  Upsampling {band_key} from {src.res[0]}m to 10m...")
                    data = src.read(
                        1,
                        out_shape=(master_height, master_width),
                        resampling=Resampling.bilinear,
                    )

                dst.write(data, i)
                dst.set_band_description(i, band_key)

    print(f"Successfully saved datacube to {out_file}")


for tile_dir in os.scandir("lab_data/S2"):
    print(f"\nProcessing directory: {tile_dir.name}")
    if tile_dir.is_dir():
        create_sentinel2_datacube(tile_dir.path, "lab_data/datacubes")

In [ ]:
import os
from pathlib import Path
import rasterio


def parse_tif_metadata(folder_path):
    folder = Path(folder_path)
    tif_files = sorted(list(folder.glob("*.tif")))

    if not tif_files:
        print(f"No .tif files found in {folder_path}")
        return

    # Table Header
    header = (
        f"{'File Name':<35} | {'Shape (H, W)':<15} | {'Bands':<6} | {'GSD (m)':<15} | {'CRS':<15}"
    )
    print("\n" + header)
    print("-" * len(header))

    for tif in tif_files:
        try:
            with rasterio.open(tif) as src:
                # Shape and Bands
                shape = f"{src.height}, {src.width}"
                bands = src.count

                # GSD (Ground Sampling Distance)
                # S2 bands vary (10m, 20m, 60m)
                # res[0] is pixel width, res[1] is pixel height
                res_x, res_y = src.res
                gsd = f"{abs(res_x):.1f} x {abs(res_y):.1f}"
                # CRS
                crs = src.crs.to_string() if src.crs else "Unknown"

                # Elegant Print
                print(f"{tif.name:<35} | {shape:<15} | {bands:<6} | {gsd:<15} | {crs:<15}")

        except Exception as e:
            print(f"{tif.name:<35} | Error: {e}")


# Run it on your S2 data directory
parse_tif_metadata("lab_data/datacubes")

# Align GT and "crop" to datacubes

In [ ]:
import os
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import from_bounds
from pathlib import Path

def create_aligned_subsets(gt_path, datacube_path, target_dir):
    """
    Finds the intersection of GT and Datacube, then outputs matching 
    subsets of both to the target_dir.
    """
    out_path = Path(target_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    
    tile_id = Path(datacube_path).name.replace("_datacube.tif", "")

    with rasterio.open(datacube_path) as cube, rasterio.open(gt_path) as gt:
        # 1. Transform GT bounds to the Datacube's CRS to find overlap
        gt_bounds_in_cube_crs = transform_bounds(gt.crs, cube.crs, *gt.bounds)
        
        # 2. Find the intersection of the two bounding boxes
        # (max of lefts, max of bottoms, min of rights, min of tops)
        int_left = max(cube.bounds.left, gt_bounds_in_cube_crs[0])
        int_bottom = max(cube.bounds.bottom, gt_bounds_in_cube_crs[1])
        int_right = min(cube.bounds.right, gt_bounds_in_cube_crs[2])
        int_top = min(cube.bounds.top, gt_bounds_in_cube_crs[3])

        if int_left >= int_right or int_bottom >= int_top:
            print(f"Skipping {tile_id}: No spatial overlap found.")
            return

        # 3. Create the Window for the subset
        window = from_bounds(int_left, int_bottom, int_right, int_top, cube.transform)
        window = window.round()
        
        # Define output paths
        subset_cube_path = out_path / f"{tile_id}_subset_datacube.tif"
        subset_gt_path = out_path / f"{tile_id}_subset_gt.tif"

        # --- A) Save Datacube Subset ---
        cube_meta = cube.meta.copy()
        cube_meta.update({
            "height": int(window.height),
            "width": int(window.width),
            "transform": cube.window_transform(window),
            "compress": "lzw"
        })

        with rasterio.open(subset_cube_path, "w", **cube_meta) as dst_cube:
            dst_cube.write(cube.read(window=window))

        # --- B) Save Aligned & Cropped GT Subset ---
        gt_meta = cube_meta.copy()
        gt_meta.update({
            "count": 1,
            "dtype": gt.dtypes[0],
            "nodata": gt.nodata
        })

        with rasterio.open(subset_gt_path, "w", **gt_meta) as dst_gt:
            reproject(
                source=rasterio.band(gt, 1),
                destination=rasterio.band(dst_gt, 1),
                src_transform=gt.transform,
                src_crs=gt.crs,
                dst_transform=dst_gt.transform,
                dst_crs=dst_gt.crs,
                resampling=Resampling.nearest
            )

    print(f"Generated matching subsets for {tile_id}:")
    print(f"  Shape: ({int(window.height)}, {int(window.width)})")

for cube in Path("lab_data/datacubes").glob("*_datacube.tif"):
    create_aligned_subsets("lab_data/GBDA24_ex2_ref_data.tif", cube, "lab_data/dataset_subsets")

In [ ]:
import rasterio
from rasterio.warp import transform_bounds
from pathlib import Path

def report_training_pairs(output_dir):
    path = Path(output_dir)
    # Find all subset datacubes to use as keys
    cubes = sorted(list(path.glob("*_subset_datacube.tif")))
    
    if not cubes:
        print(f"No processed subsets found in {output_dir}")
        return

    header = f"{'File Name':<35} | {'Shape (H, W)':<15} | {'Bands':<6} | {'GSD (m)':<10} | {'CRS':<10} | {'Bounds (WGS84)'}"
    print("\n" + header)
    print("-" * 140)

    for cube_path in cubes:
        # Identify the matching GT file
        tile_id = cube_path.name.replace("_subset_datacube.tif", "")
        gt_path = path / f"{tile_id}_subset_gt.tif"
        
        # Report for the Datacube and the GT back-to-back
        for f_path in [cube_path, gt_path]:
            if not f_path.exists():
                continue
                
            with rasterio.open(f_path) as src:
                # 1. Basic Stats
                shape = f"{src.height}, {src.width}"
                bands = src.count
                gsd = f"{abs(src.res[0]):.1f}"
                crs = src.crs.to_epsg()
                
                # 2. Calculate WGS84 Bounds
                # transform_bounds converts from the file's CRS to EPSG:4326
                wgs_bounds = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
                bounds_str = f"L: {wgs_bounds[0]:.4f}, B: {wgs_bounds[1]:.4f}, R: {wgs_bounds[2]:.4f}, T: {wgs_bounds[3]:.4f}"
                
                # 3. Print Row
                print(f"{f_path.name:<35} | {shape:<15} | {bands:<6} | {gsd:<10} | EPSG:{crs:<5} | {bounds_str}")
        
        # Add a small separator between pairs for readability
        print("-" * 140)

report_training_pairs("lab_data/dataset_subsets")